# Three methods with NumPy/SciPy and JAX

Compare **BM4Implicit** (symmetric projection), **GaussLegendre4** (two stages, fourth order) and **RK2** (explicit midpoint) using the union of sets 3 and 4 (16 particles) and the reflected reference potential at normalized gyro-radius **0.015**. Integrate 10, 30 and 90 cycles with 40 fixed steps and ten saved samples per cycle.

A fresh run executes 72 integrations: one first call and three later calls for each library in each of the nine cases. Both libraries run sequentially in the same Modal CPU container for each case (two physical cores, 4 GiB requested / 8 GiB limit). Sources and numerical inputs are frozen before submission. Existing confirmed calls are recovered by their receipts; completed trajectories are reused. Choose a new run identifier to measure again.

Use the project `.venv` kernel. Scientific results belong to the explicit B2 destination below; `outputs/` is a recovery cache. The previous two-projection BM4 study remains archived under `bm4-20261117-rho0015-simple`; none of its timings substitutes a new measurement.


In [1]:
from pathlib import Path
import json
import sys

# Locate the code repository from this section folder or its parent directories.
ROOT = next(
    (candidate.resolve()
     for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     for candidate in (parent, parent / "GC2D_codigo")
     if (candidate / "src/studies/bm4_numpy_jax.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "GC2D_codigo was not found. Keep it beside TFM_redaccion "
        "or start the notebook from the code repository."
    )
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

STUDY_PREFIX = "studies/bm4_numpy_jax"
RUN_ID = "methods-20261119-union16-rho0015-c10-30-90"
RUN_DIRECTORY = ROOT / "outputs" / STUDY_PREFIX / RUN_ID
BUCKET = f"gc2d_data:gc2d-notebooks-data/{STUDY_PREFIX}/{RUN_ID}"


## Inputs and timing scope

The study checks the source HDF5 fingerprint and both eight-particle selections against the 84-particle record, and reads the representative radius from `TFM_redaccion/datos/campana_referencia.json`. It uses the same field indices, normalization and analytic Newton controls as the reference campaign: absolute tolerance `1e-12`, relative tolerance `1e-11`, at most 40 corrections.

Each timer covers `Execution.run`: preparation, integration, synchronization and NumPy result construction. Input restoration, disk/network access, audits and plotting are excluded. No observers or progress reporting are enabled. `record_step_statistics=False` avoids collecting step histories while keeping finite-state and convergence checks. The first JAX call must compile; each later call must reuse its executable. Library order alternates, with bitwise repetition checks outside the timed interval. Backend agreement uses Euclidean distance in unwrapped base-cell coordinates and does not establish integration accuracy.

The protocol follows the [JAX benchmarking guidance](https://docs.jax.dev/en/latest/benchmarking.html).

In [2]:
from studies.bm4_numpy_jax_io import prepare_study, run_study
from diagnostics.io.storage import ArtifactStore

CYCLES = (10, 30, 90)
REPETITIONS = 3
if not (RUN_DIRECTORY / "study.json").exists():
    store = ArtifactStore(BUCKET)
    if store.exists("publication.json"):
        RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
        for name in ("study.json", "publication.json"):
            store.get(name, RUN_DIRECTORY / name)
    else:
        prepare_study(ROOT, RUN_DIRECTORY, cycles=CYCLES, repetitions=REPETITIONS)
study = json.loads((RUN_DIRECTORY / "study.json").read_text())
assert study["cycles"] == list(CYCLES) and study["repetitions"] == REPETITIONS
assert study["methods"] == ["BM4Implicit", "GaussLegendre4", "RK2"]
assert study["bucket"] == BUCKET
assert study["particle_count"] == 16
print(json.dumps(study, indent=2))


{
  "schema_version": 2,
  "study_prefix": "studies/bm4_numpy_jax",
  "run_id": "methods-20261119-union16-rho0015-c10-30-90",
  "cycles": [
    10,
    30,
    90
  ],
  "methods": [
    "BM4Implicit",
    "GaussLegendre4",
    "RK2"
  ],
  "repetitions": 3,
  "particle_count": 16,
  "particle_ids": [
    1,
    2,
    12,
    15,
    18,
    21,
    24,
    27,
    30,
    33,
    60,
    64,
    65,
    72,
    76,
    83
  ],
  "particle_selection": {
    "operation": "union",
    "source_sets": [
      {
        "dataset_id": "chaotic_candidates",
        "source_file": "chaotic_candidates.json",
        "sha256": "493656ec43af248c63bf0081190fae19c20026c3188e99c648587626d5b29133",
        "particle_ids": [
          12,
          15,
          18,
          21,
          24,
          27,
          30,
          33
        ]
      },
      {
        "dataset_id": "periodic_candidates_8",
        "source_file": "periodic_candidates_8.json",
        "sha256": "6bc821e9166c2304f9277fa

## Calculate, recover and archive

This cell submits missing jobs only. A saved successful publication is reused. Failed or interrupted transfers keep the local results; rerunning recovers confirmed calls without restarting their integrations. A failed numerical call remains explicit and requires investigation; it is never silently replaced.

The remote result includes the individual first/warmed times, CPU and library versions, paired trajectories and trajectory discrepancies. Each backend is saved through `diagnostics.io.persistence`; B2 publication is verified by loading the archives again and comparing their states and provenance.
If no local record exists, the preceding cell first checks for an already published run in B2. Opening this notebook on another machine therefore recovers the published results without submitting them again.


In [3]:
publication_path = RUN_DIRECTORY / "publication.json"
if publication_path.exists():
    publication = json.loads(publication_path.read_text())
    assert publication["status"] == "verified" and publication["bucket"] == BUCKET
    print(f"Reusing the completed run: {RUN_ID}")
else:
    run_study(ROOT, RUN_DIRECTORY, publish=True)
    publication = json.loads(publication_path.read_text())
print(json.dumps(publication, indent=2))

Reusing the completed run: methods-20261119-union16-rho0015-c10-30-90
{
  "status": "verified",
  "bucket": "gc2d_data:gc2d-notebooks-data/studies/bm4_numpy_jax/methods-20261119-union16-rho0015-c10-30-90",
  "verified_cases": [
    "BM4Implicit-10/scipy",
    "BM4Implicit-10/jax",
    "GaussLegendre4-10/scipy",
    "GaussLegendre4-10/jax",
    "RK2-10/scipy",
    "RK2-10/jax",
    "BM4Implicit-30/scipy",
    "BM4Implicit-30/jax",
    "GaussLegendre4-30/scipy",
    "GaussLegendre4-30/jax",
    "RK2-30/scipy",
    "RK2-30/jax",
    "BM4Implicit-90/scipy",
    "BM4Implicit-90/jax",
    "GaussLegendre4-90/scipy",
    "GaussLegendre4-90/jax",
    "RK2-90/scipy",
    "RK2-90/jax"
  ]
}


In [4]:
from visualization.bm4_numpy_jax import summarize

summary = summarize(RUN_DIRECTORY, bucket=BUCKET)
for row in summary["rows"]:
    print(f"{row['method']:12s} {row['cycles']:4d} cycles | "
          f"NumPy/SciPy {row['backends']['scipy']['median_seconds']:.3f} s | "
          f"JAX {row['backends']['jax']['median_seconds']:.3f} s | "
          f"speedup {row['speedup']:.2f} | max distance {row['maximum_distance']:.3e}")

BM4Implicit    10 cycles | NumPy/SciPy 20.360 s | JAX 1.248 s | speedup 16.31 | max distance 8.162e-14
GaussLegendre4   10 cycles | NumPy/SciPy 2.690 s | JAX 0.087 s | speedup 30.80 | max distance 4.947e-14
RK2            10 cycles | NumPy/SciPy 0.417 s | JAX 0.015 s | speedup 27.52 | max distance 2.706e-14
BM4Implicit    30 cycles | NumPy/SciPy 59.096 s | JAX 2.835 s | speedup 20.84 | max distance 1.824e-12
GaussLegendre4   30 cycles | NumPy/SciPy 7.918 s | JAX 0.201 s | speedup 39.45 | max distance 1.061e-12
RK2            30 cycles | NumPy/SciPy 1.351 s | JAX 0.038 s | speedup 35.30 | max distance 1.181e-12
BM4Implicit    90 cycles | NumPy/SciPy 182.845 s | JAX 10.646 s | speedup 17.18 | max distance 1.837e-06
GaussLegendre4   90 cycles | NumPy/SciPy 26.397 s | JAX 0.738 s | speedup 35.78 | max distance 1.025e-06
RK2            90 cycles | NumPy/SciPy 3.554 s | JAX 0.173 s | speedup 20.55 | max distance 7.554e-07


Open **visualisation.ipynb** to regenerate both thesis figures from these saved results. Plot changes require no trajectory integrations.